# Cómo usar este notebook

> Este notebook no está pensado solo para analistas. Está diseñado para ayudar a convertir preguntas de negocio en decisiones concretas sobre `conf/settings.yaml`.

## Qué vas a encontrar aquí

- Una explicación paso a paso de los parámetros más importantes del proyecto.
- Bloques de análisis que responden preguntas concretas: cuántos datos hacen falta, qué variables conviene dejar, qué señales son redundantes, qué umbral es prudente y cómo priorizar clientes.
- Una traducción práctica desde la salida estadística hacia decisiones simples de configuración.

## Cómo leerlo si no eres técnico

1. Lee el título y la explicación de cada sección antes de mirar el código.
2. Fíjate en las frases que empiezan por "pregunta de negocio", "qué decisión ayuda a tomar" y "qué mover en settings.yaml".
3. Si una salida parece muy técnica, ve directo al bloque de interpretación que aparece justo después.
4. No hace falta entender cada fórmula para sacar valor: este notebook está pensado para que puedas entender la conclusión y la acción recomendada.

## Importante antes de empezar

- Las primeras celdas técnicas solo preparan el entorno del notebook.
- Si ves imports, parches de compatibilidad o funciones auxiliares, puedes tratarlos como "infraestructura" para que el análisis funcione.
- El valor real para lectura de negocio empieza en las secciones numeradas.

In [1]:
# Parche: asegurar que `proportion_effectsize` esté disponible en
# `statsmodels.stats.power` para compatibilidad con diferentes versiones.
import sys
import importlib
import types
import numpy as np
try:
    from statsmodels.stats.proportion import proportion_effectsize
except Exception:
    def proportion_effectsize(p1, p2):
        # Cohen's h (transformación arcsin), devuelve valor absoluto
        return abs(2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2)))
try:
    pwr = importlib.import_module("statsmodels.stats.power")
    setattr(pwr, "proportion_effectsize", proportion_effectsize)
except Exception:
    pwr = types.ModuleType("statsmodels.stats.power")
    pwr.proportion_effectsize = proportion_effectsize
    sys.modules["statsmodels.stats.power"] = pwr
print('Parche aplicado: proportion_effectsize listo en statsmodels.stats.power')


Parche aplicado: proportion_effectsize listo en statsmodels.stats.power


### Interpretación — Parche de compatibilidad

- Propósito: esta celda asegura que la función `proportion_effectsize` esté disponible independientemente de la versión de `statsmodels` que tenga el kernel.
- Qué esperar: imprime `Parche aplicado: proportion_effectsize listo en statsmodels.stats.power` si se aplica correctamente.
- Uso práctico: no requiere cambios para ejecutar el notebook; si ves errores de importación posteriores, comprueba que el kernel del notebook use el mismo `venv` donde instalaste paquetes.
- Solución rápida si falla: reinicia el kernel o instala/actualiza `statsmodels` en el intérprete activo:

```powershell
# desde PowerShell (ajusta la ruta al python de tu venv)
& ".\.venv\Scripts\python.exe" -m pip install -U statsmodels
```


# Guía práctica para parametrizar `conf/settings.yaml`
Este notebook contiene snippets reproducibles para: análisis de potencia, VIF, selección de features (RFE), parsimonia (AIC/BIC), detección de outliers, validación temporal purgada, cálculo de CLV para distintos `clv_horizons`, cálculo de `retention_success_rate` con bootstrap y escalado de `segment_features`.

Cada fragmento es autocontenido y está pensado para adaptarse al dataset de `data/raw/` del proyecto.

## Mapa rápido de decisiones

> Si solo quieres saber qué parámetro toca cada bloque, usa esta tabla como guía.

| Sección | Pregunta que responde | Parámetro(s) que ayuda a decidir |
| --- | --- | --- |
| 2. Análisis de potencia | ¿Tenemos suficientes datos para defender una conclusión? | `case.dataset_rows` |
| 3. VIF | ¿Hay variables que cuentan casi la misma historia? | `case.vif_cols` |
| 4. RFE | ¿Qué variables conviene dejar para explicar el fenómeno? | `case.inference_features` |
| 5. Parsimonia | ¿Necesitamos un modelo más complejo o uno más simple basta? | `case.parsimony.*`, revisión de `inference_features` |
| 6. Outliers | ¿Qué columnas tienen valores extremos que pueden distorsionar el análisis? | `case.outlier_cols` |
| 7. Validación temporal | ¿La métrica aguanta cuando respetamos el tiempo real del negocio? | `case.temporal_validation.*` |
| 8. CLV | ¿Qué horizonte de valor futuro tiene sentido usar? | `case.clv_horizons` |
| 9. Retention success rate | ¿Qué tasa de éxito usar para estimar valor esperado? | `case.retention_success_rate` |
| 10. Segment features | ¿Qué variables deben alimentar la segmentación operativa? | `case.segment_features` |
| 11. Editar YAML | ¿Cómo llevar estas decisiones al archivo de configuración? | `conf/settings.yaml` |

## Regla de lectura sencilla

- Si una sección termina con una recomendación concreta, esa recomendación pesa más que el detalle matemático.
- Si una métrica y la lógica de negocio se contradicen, la decisión final debe justificarse, no automatizarse a ciegas.
- El objetivo no es "ganar una métrica" sino dejar un `settings.yaml` defendible ante negocio, auditoría y operación.

## 1) Setup: librerías requeridas

> Esta sección no toma decisiones de negocio. Solo prepara el entorno para que las siguientes celdas funcionen sin errores.

**Pregunta de negocio que resuelve:** ninguna todavía.

**Qué hace realmente:** carga las librerías estadísticas, de machine learning y de configuración que se usan a lo largo del notebook.

**Qué necesitas saber si no eres técnico:**

- Si esta parte falla, el resto del notebook no podrá ejecutarse bien.
- Si funciona, puedes continuar con tranquilidad aunque no entiendas todos los imports.

**Qué decisión ayuda a tomar:** confirmar que el entorno es consistente antes de confiar en cualquier recomendación posterior.

In [2]:
import numpy as np
import pandas as pd
# Estadística y selección
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.power import NormalIndPower, TTestIndPower
# Importar proportion_effectsize desde el módulo correcto cuando esté disponible,
# y proporcionar una implementación de reserva si la versión de statsmodels no lo incluye.
try:
    from statsmodels.stats.proportion import proportion_effectsize
except Exception:
    def proportion_effectsize(p1, p2):
        # Cohen's h
        h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))
        return abs(h)
# ML
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
# CLV (opcional)
try:
    from lifetimes import BetaGeoFitter, GammaGammaFitter
except Exception:
    BetaGeoFitter = None
    GammaGammaFitter = None
# YAML
import yaml


### Interpretación — Setup de librerías e importaciones

- Propósito: importa las librerías usadas por las celdas siguientes y define implementaciones de reserva para compatibilidad.
- Qué comprobar si algo falla:
  - `sys.executable` debe apuntar al `python` del `venv` del proyecto.
  - Si aparece `ImportError`, instala/actualiza el paquete en el mismo intérprete del kernel.
- No produce una tabla de salida, pero prepara funciones (ej. `compute_vif`, `rfe_select`) usadas más adelante.
- Comandos útiles (ejecutar en PowerShell con la ruta correcta al `python` del venv):

```powershell
& ".\.venv\Scripts\python.exe" -m pip install -r requirements.txt
& ".\.venv\Scripts\python.exe" -m pip install -e .
```


---

## 2) Análisis de potencia (Power Analysis)

> Aquí respondemos una pregunta simple pero crítica: **¿tenemos suficientes datos para sacar una conclusión creíble?**

**Pregunta de negocio:** ¿el volumen actual de datos alcanza para detectar una mejora o una diferencia que realmente importe?

**Qué parámetro ayuda a decidir:** `case.dataset_rows`.

**Cómo leer esta sección si no eres técnico:**

- Si la recomendación pide más filas, no significa que el modelo sea malo. Significa que todavía no hay suficiente evidencia para afirmar algo con seguridad.
- Si el tamaño actual sí alcanza, puedes defender que el análisis tiene base estadística razonable.

**Qué decisión ayuda a tomar:** aumentar el tamaño de muestra, redefinir el efecto mínimo que quieres detectar o ajustar expectativas sobre la potencia del análisis.

In [16]:
# Power analysis mejorado: calcula el tamaño de muestra requerido usando el dataset bank360_dataset.csv
import math
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from statsmodels.stats.power import NormalIndPower, TTestIndPower

# Asegurar proportion_effectsize disponible (viene del setup, pero fallback local)
try:
    proportion_effectsize
except NameError:
    try:
        from statsmodels.stats.proportion import proportion_effectsize
    except Exception:
        def proportion_effectsize(p1, p2):
            return abs(2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2)))

# Cargar dataset (buscar en rutas habituales)
candidates = [
    "data/raw/bank360_dataset.csv",
    "../data/raw/bank360_dataset.csv",
    "banca_360_metodologia_v3/data/raw/bank360_dataset.csv",
    "../banca_360_metodologia_v3/data/raw/bank360_dataset.csv",
    "Notebooks/data/raw/bank360_dataset.csv",
]

data_path = None
for p in candidates:
    if Path(p).exists():
        data_path = Path(p)
        break
if data_path is None:
    raise FileNotFoundError("No se encontró bank360_dataset.csv en rutas habituales: " + ", ".join(candidates))

df = pd.read_csv(data_path)
N = len(df)
print(f"Dataset cargado: {data_path} → {N} filas")

# Parámetros por defecto (ajustables)
p1 = 0.20  # tasa actual (ej. retención)
p2 = 0.25  # tasa mínima detectable (negocio)
alpha = 0.05
power = 0.8
buffer_pct = 0.10  # margen adicional para pérdidas/NA

# Cálculo para proporciones (2 grupos, tamaño por grupo)
effect = proportion_effectsize(p1, p2)
analysis = NormalIndPower()
n_per_group = math.ceil(analysis.solve_power(effect_size=effect, power=power, alpha=alpha, ratio=1))
total_needed = n_per_group * 2
recommended_total = math.ceil(total_needed * (1 + buffer_pct))

print('\n--- Proporciones (comparación 2 grupos) ---')
print(f'Tasa actual (p1) = {p1}, tasa mínima detectable (p2) = {p2}')
print(f'Tamaño requerido por grupo: {n_per_group}')
print(f'Tamaño total requerido (2 grupos): {total_needed}')
print(f'Recomendación (con +{int(buffer_pct*100)}% buffer): {recommended_total}')

if N >= recommended_total:
    print(f"OK: el dataset actual ({N}) es suficiente para detectar esa diferencia.")
else:
    deficit = recommended_total - N
    print(f"AVISO: el dataset actual ({N}) es insuficiente por {deficit} filas.\nConsidera recolectar más datos o ajustar p2/power/buffer.)")

# Comparación con valor en settings.yaml si existe
settings_candidates = ["conf/settings.yaml", "settings.yaml", "notebooks/settings.yaml", "Notebooks/settings.yaml"]
settings_path = None
for s in settings_candidates:
    if Path(s).exists():
        settings_path = Path(s)
        try:
            cfg = yaml.safe_load(settings_path.read_text(encoding='utf-8'))
            current_dataset_rows = cfg.get('case', {}).get('dataset_rows')
        except Exception:
            current_dataset_rows = None
        break
else:
    current_dataset_rows = None

if current_dataset_rows is not None:
    print(f"\nValor actual en settings.yaml (case.dataset_rows): {current_dataset_rows}")
    if current_dataset_rows >= recommended_total:
        print("El valor actual en settings.yaml es suficiente según este análisis.")
    else:
        print(f"Sugerencia: aumenta `case.dataset_rows` a al menos {recommended_total} para este objetivo.")
else:
    print('\nNo se encontró un `case.dataset_rows` legible en los YAMLs buscados.')

# Snippet recomendado para settings.yaml
print('\nSnippet sugerido para settings.yaml:')
print('case:')
print(f'  dataset_rows: {recommended_total}')

# También evaluar diferencia de medias (Cohen\'s d)
effect_size = 0.2  # Cohen's d (ajustable)
analysis_t = TTestIndPower()
n_per_group_t = math.ceil(analysis_t.solve_power(effect_size=effect_size, alpha=alpha, power=power))
total_needed_t = n_per_group_t * 2
recommended_total_t = math.ceil(total_needed_t * (1 + buffer_pct))

print('\n--- Diferencia de medias (Cohen\'s d) ---')
print(f'Effect size (d) = {effect_size}')
print(f'Tamaño por grupo: {n_per_group_t}   → total: {total_needed_t}')
print(f'Recomendación (con +{int(buffer_pct*100)}% buffer): {recommended_total_t}')
if N >= recommended_total_t:
    print(f"OK: el dataset actual ({N}) es suficiente para detectar ese efecto de medias.")
else:
    deficit2 = recommended_total_t - N
    print(f"AVISO: el dataset actual ({N}) es insuficiente por {deficit2} filas para detectar ese efecto de medias.")


Dataset cargado: data\raw\bank360_dataset.csv → 1200 filas

--- Proporciones (comparación 2 grupos) ---
Tasa actual (p1) = 0.2, tasa mínima detectable (p2) = 0.25
Tamaño requerido por grupo: 1092
Tamaño total requerido (2 grupos): 2184
Recomendación (con +10% buffer): 2403
AVISO: el dataset actual (1200) es insuficiente por 1203 filas.
Considera recolectar más datos o ajustar p2/power/buffer.)

Valor actual en settings.yaml (case.dataset_rows): 1200
Sugerencia: aumenta `case.dataset_rows` a al menos 2403 para este objetivo.

Snippet sugerido para settings.yaml:
case:
  dataset_rows: 2403

--- Diferencia de medias (Cohen's d) ---
Effect size (d) = 0.2
Tamaño por grupo: 394   → total: 788
Recomendación (con +10% buffer): 867
OK: el dataset actual (1200) es suficiente para detectar ese efecto de medias.


### Interpretación — Análisis de potencia

> Traducción simple: esta salida te dice si la cantidad de datos alcanza para defender una decisión con suficiente respaldo estadístico.

- Si el tamaño recomendado es mayor que tu muestra actual, la conclusión correcta no es "seguir igual", sino reconocer que todavía falta evidencia.
- Si el tamaño actual sí supera la recomendación, puedes defender mejor que el análisis tiene base suficiente.
- En `settings.yaml`, esta sección impacta sobre todo en `case.dataset_rows`.

**Señal de alerta para no técnicos:** si necesitas muchas más filas de las que hoy tienes, evita presentar resultados como definitivos.

In [4]:
# Diferencia de medias (ej. ingreso medio)
from statsmodels.stats.power import TTestIndPower
effect_size = 0.2  # d de Cohen: pequeño=0.2, medio=0.5, grande=0.8
alpha = 0.05
power = 0.8
analysis = TTestIndPower()
n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='two-sided')
print(f'Necesitas ~{int(np.ceil(n))} observaciones por grupo (two-sided t-test)')

Necesitas ~394 observaciones por grupo (two-sided t-test)


### Interpretación — Diferencia de medias

- Salida: `n` por grupo para detectar el `effect_size` (Cohen's d). Interpreta `effect_size` con criterios de negocio (0.2 pequeño, 0.5 medio, 0.8 grande).
- Acción: si `n` es inviables, considerar test emparejado, aumentar el efecto mínimo detectable o reducir la potencia objetivo.
- Consejo: siempre vincula `effect_size` a un impacto económico verificable (p. ej. cambio en ingreso medio que justifique intervención).

---

## 3) VIF (multicolinealidad)

> Esta sección ayuda a detectar cuándo varias variables están contando casi la misma historia y pueden volver inestable la interpretación del modelo.

**Pregunta de negocio:** ¿estamos usando variables redundantes que complican el modelo sin aportar información nueva?

**Qué parámetro ayuda a decidir:** `case.vif_cols`.

**Cómo leer esta sección si no eres técnico:**

- Un VIF alto no significa automáticamente que una variable esté mal.
- Sí significa que algunas variables pueden estar demasiado conectadas entre sí y hacer más frágil la lectura del modelo.

**Qué decisión ayuda a tomar:** qué variables revisar, combinar o simplificar antes de pasar a producción.

In [5]:
def compute_vif(df, vif_cols):
    X = df[vif_cols].copy()
    X = sm.add_constant(X)
    vifs = {}
    for i, col in enumerate(vif_cols, start=1):
        try:
            vifs[col] = variance_inflation_factor(X.values, i)
        except Exception as e:
            vifs[col] = np.nan
    return vifs
# Uso: vifs = compute_vif(df, ['edad','ingreso_mensual','saldo_promedio_3m'])

In [14]:
# Ejecutar VIF usando `conf/settings.yaml` y el dataset disponible
import os
from pathlib import Path
import yaml
import pandas as pd

# Buscar settings.yaml en rutas comunes
cfg = None
settings_path = None
for p in ("conf/settings.yaml", "settings.yaml", "../conf/settings.yaml", "../settings.yaml"):
    if Path(p).exists():
        settings_path = p
        with open(p, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f)
        break
if cfg is None:
    raise FileNotFoundError("No se encontró conf/settings.yaml en rutas habituales.")

vif_cols = cfg.get("case", {}).get("vif_cols", [])
print("Settings cargado:", settings_path)
print("VIF candidate cols:", vif_cols)

# Buscar dataset en rutas comunes
df = None
data_path = None
candidates = [
    "data/raw/bank360_dataset.csv",
    "data/raw/bank360_dataset.csv",
    "../data/raw/bank360_dataset.csv",
    "banca_360_metodologia_v3/data/raw/bank360_dataset.csv",
    "../banca_360_metodologia_v3/data/raw/bank360_dataset.csv",
]
for p in candidates:
    if Path(p).exists():
        data_path = p
        df = pd.read_csv(p)
        break
if df is None:
    raise FileNotFoundError("No se encontró bank360_dataset.csv en rutas habituales.")
print("Dataset cargado:", data_path, "->", df.shape)

# Filtrar columnas disponibles para VIF
vif_cols_present = [c for c in vif_cols if c in df.columns]
print("VIF cols presentes:", vif_cols_present)
if not vif_cols_present:
    raise ValueError("Ninguna columna de VIF está presente en el dataset.")

# Convertir a numérico y eliminar filas con NA en esas columnas
X = df[vif_cols_present].apply(pd.to_numeric, errors='coerce')
X = X.dropna()
print("Datos después de coerción y dropna:", X.shape)

# Ejecutar compute_vif (definido en el notebook previamente)
vifs = compute_vif(X, vif_cols_present)
import pandas as pd
vif_df = pd.Series(vifs).sort_values(ascending=False).rename("VIF").to_frame()
print('\nVIF por variable:')
display(vif_df)


Settings cargado: conf/settings.yaml
VIF candidate cols: ['edad', 'antiguedad_meses', 'ingreso_mensual', 'saldo_promedio_3m', 'gasto_mensual', 'ingreso_mensual_fracdiff_0_4', 'indice_engagement']
Dataset cargado: data/raw/bank360_dataset.csv -> (1200, 42)
VIF cols presentes: ['edad', 'antiguedad_meses', 'ingreso_mensual', 'saldo_promedio_3m', 'gasto_mensual', 'ingreso_mensual_fracdiff_0_4', 'indice_engagement']
Datos después de coerción y dropna: (156, 7)

VIF por variable:


,VIF
ingreso_mensual,10.242594
ingreso_mensual_fracdiff_0_4,8.303459
gasto_mensual,1.930935
saldo_promedio_3m,1.878792
indice_engagement,1.088488
antiguedad_meses,1.080090
edad,1.044099


### Interpretación — VIF (multicolinealidad)

> Traducción simple: este bloque detecta si hay variables demasiado parecidas entre sí para una lectura estable del modelo.

- Un VIF alto sugiere que dos o más variables están empujando en la misma dirección y pueden volver frágiles los coeficientes.
- No significa necesariamente eliminar variables de inmediato; significa revisarlas con criterio.
- En `settings.yaml`, esta sección ayuda a revisar `case.vif_cols` y, si hace falta, limpiar también `case.inference_features`.

**Regla práctica para negocio:** si varias variables cuentan casi la misma historia, suele convenir quedarse con la más interpretable o accionable.

---

## 4) Selección de features (RFE) para `inference_features`

> Esta parte busca responder qué variables conviene conservar para explicar el fenómeno sin cargar el modelo con ruido innecesario.

**Pregunta de negocio:** ¿qué variables vale la pena dejar para explicar y predecir abandono de forma razonable?

**Qué parámetro ayuda a decidir:** `case.inference_features`.

**Cómo leer esta sección si no eres técnico:**

- No interpreta "la verdad absoluta" sobre las variables más importantes.
- Interpreta una propuesta inicial de variables útiles que luego debe contrastarse con negocio, estabilidad temporal y sentido operativo.

**Qué decisión ayuda a tomar:** qué variables se quedan como base explicativa del modelo y cuáles deberían salir del contrato.

In [6]:
def rfe_select(X, y, n_features=10, random_state=42):
    model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=random_state)
    selector = RFE(model, n_features_to_select=n_features, step=1)
    selector = selector.fit(X, y)
    selected = X.columns[selector.support_].tolist()
    ranking = dict(zip(X.columns, selector.ranking_))
    return selected, ranking
# Uso: selected, ranking = rfe_select(X_train, y_train, n_features=12)

In [17]:
# Ejecutar RFE (Recursive Feature Elimination) y mostrar salida didáctica
from pathlib import Path
import yaml, pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, cross_val_score

# localizar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml","banca_360_metodologia_v3/conf/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        settings_path = sp
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

print(f"Dataset: {data_path} | filas: {len(df)} | settings: {settings_path}")

target = cfg['case'].get('target')
feature_cols = [c for c in cfg['case'].get('feature_cols', []) if c in df.columns]
if target not in df.columns:
    raise ValueError(f"Target '{target}' no encontrado en el dataset")

# preparar X, y (codificación simple)
X = df[feature_cols].copy()
for col in X.select_dtypes(include=['object','category']).columns:
    if X[col].nunique() <= 10:
        dummies = pd.get_dummies(X[col].astype(str), prefix=col, drop_first=True)
        X = pd.concat([X.drop(columns=col), dummies], axis=1)
    else:
        X[col] = X[col].astype('category').cat.codes

# preparar y robusto

def prepare_target(s):
    import pandas as _pd
    if _pd.api.types.is_numeric_dtype(s):
        return s.astype(int)
    s2 = s.astype(str).str.strip().str.lower()
    mapvals = {'si':1,'sí':1,'s':1,'1':1,'true':1,'yes':1,'y':1,'no':0,'n':0,'0':0,'false':0}
    mapped = s2.map(mapvals)
    if mapped.isna().any():
        num = pd.to_numeric(s, errors='coerce')
        if num.notna().all():
            return num.astype(int)
    return mapped

y = prepare_target(df[target])
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int)

# limpieza columnas constantes y NA
X = X.dropna(axis=1, how='all')
X = X.loc[:, X.nunique() > 1]

if X.shape[0] < 40:
    print('Advertencia: pocas observaciones después del filtrado:', X.shape)

# definir n_features a seleccionar
inference_len = len(cfg['case'].get('inference_features', [])) or min(12, X.shape[1])
n_select = min(inference_len, X.shape[1], 12)

est = LogisticRegression(max_iter=1000, solver='liblinear')
selector = RFE(est, n_features_to_select=n_select, step=1)
selector = selector.fit(X.fillna(0), y)
selected = list(X.columns[selector.support_])
print('\nRFE - features seleccionados:')
for i,f in enumerate(selected,1):
    print(f"  {i}. {f}")

# rankings
rank = pd.Series(selector.ranking_, index=X.columns).sort_values()
print('\nRanking (1=seleccionado):')
print(rank.head(30))

# evaluación rápida (CV AUC)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
try:
    aucs = cross_val_score(LogisticRegression(max_iter=1000, solver='liblinear'), X[selected].fillna(0), y, scoring='roc_auc', cv=cv, n_jobs=-1)
    print(f"\nCV ROC AUC (RFE-selected): mean={aucs.mean():.4f} std={aucs.std():.4f}")
except Exception as e:
    print('No se pudo calcular CV AUC:', e)

print('\nSugerencia para settings.yaml:')
print('case:')
print(f"  inference_features: {selected}")

# breve nota didáctica
print('\nInterpretación:')
print('- Las features listadas arriba son las seleccionadas por RFE usando un estimador logístico simple.')
print('- Revise correlaciones y colinealidad (VIF) antes de aprobar las features para producción.')
print('- Si algunas variables son categóricas con muchas categorías, considere agrupar o crear embeddings.')


Dataset: data/raw/bank360_dataset.csv | filas: 1200 | settings: conf/settings.yaml

RFE - features seleccionados:
  1. saldo_variacion_pct_3m
  2. productos_activos
  3. ratio_uso_credito
  4. reclamaciones
  5. atraso_30d
  6. usa_app_Si
  7. nomina_domiciliada_Si
  8. hipoteca_activa_Si
  9. producto_premium_Si
  10. macro_segmento_Premium
  11. macro_segmento_Recuperacion
  12. macro_segmento_Valor

Ranking (1=seleccionado):
saldo_variacion_pct_3m             1
productos_activos                  1
ratio_uso_credito                  1
producto_premium_Si                1
nomina_domiciliada_Si              1
usa_app_Si                         1
atraso_30d                         1
macro_segmento_Valor               1
macro_segmento_Recuperacion        1
macro_segmento_Premium             1
hipoteca_activa_Si                 1
reclamaciones                      1
canal_servicio_Mixto               2
satisfaccion                       3
canal_servicio_Sucursal            4
transacciones

### Interpretación — Selección RFE

> Traducción simple: esta salida propone un conjunto inicial de variables que, juntas, explican mejor el problema que otras alternativas.

- Toma `selected` como una propuesta razonable, no como una orden automática.
- Si una variable sale bien en RFE pero no tiene sentido de negocio, conviene discutirla antes de promoverla.
- Si una variable es muy importante pero operacionalmente no es usable, también debe cuestionarse.
- En `settings.yaml`, el bloque impacta sobre `case.inference_features`.

**Criterio sano:** una variable útil en producción no solo predice; también debe poder explicarse y mantenerse.

---

## 5) Parsimonia: comparar AIC/BIC en modelos logísticos anidados

> Aquí no buscamos el modelo más sofisticado, sino el más defendible: el que mantiene un buen rendimiento con la menor complejidad razonable.

**Pregunta de negocio:** ¿realmente necesitamos muchas variables o un modelo más simple ya explica casi lo mismo?

**Qué parámetro ayuda a decidir:** `case.parsimony.logistic_feature_steps`, `case.parsimony.roc_auc_tolerance` y, en la práctica, una revisión de `case.inference_features`.

**Cómo leer esta sección si no eres técnico:**

- Si un modelo más simple pierde muy poco desempeño, suele ser preferible por claridad, estabilidad y facilidad de mantenimiento.
- Si el modelo complejo mejora mucho, entonces sí puede justificarse la complejidad adicional.

**Qué decisión ayuda a tomar:** cuántas variables conservar y qué nivel de complejidad es razonable para producción.

In [7]:
def compare_nested_logistic(y, X, feature_steps):
    results = []
    for k in feature_steps:
        X_sub = X.iloc[:, :k]  # asume columnas ordenadas por importancia previa
        try:
            model = sm.Logit(y, sm.add_constant(X_sub)).fit(disp=False)
            results.append({'k': k, 'aic': model.aic, 'bic': model.bic})
        except Exception as e:
            results.append({'k': k, 'aic': np.nan, 'bic': np.nan})
    return pd.DataFrame(results)
# Uso: compare_nested_logistic(y_train, X_train[selected_features], feature_steps=[4,8,12,18])

In [18]:
# Parsimonia (AIC/BIC) — ejecución didáctica
import numpy as np
import pandas as pd
from pathlib import Path
import yaml
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

# Cargar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml","banca_360_metodologia_v3/conf/settings.yaml"]
cfg = None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('No se encontró settings.yaml')

# Dataset
data_candidates = ['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv','../data/raw/bank360_dataset.csv']
df = None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path = dp
        break
if df is None:
    raise FileNotFoundError('No se encontró bank360_dataset.csv')

target = cfg['case'].get('target')
feature_cols = [c for c in cfg['case'].get('feature_cols', []) if c in df.columns]
parsimony_steps = cfg['case'].get('parsimony', {}).get('logistic_feature_steps', [4,8,12])
roc_tol = cfg['case'].get('parsimony', {}).get('roc_auc_tolerance', 0.015)
seed = cfg.get('seed', 42)

print(f"Dataset: {data_path} rows={len(df)} | target='{target}' | candidate features={len(feature_cols)}")
if target not in df.columns:
    raise ValueError('Target no encontrado en dataset')

# Preprocess features: simple encoding as in RFE
X = df[feature_cols].copy()
for col in X.select_dtypes(include=['object','category']).columns:
    if X[col].nunique() <= 10:
        dummies = pd.get_dummies(X[col].astype(str), prefix=col, drop_first=True)
        X = pd.concat([X.drop(columns=col), dummies], axis=1)
    else:
        X[col] = X[col].astype('category').cat.codes

# Prepare y
def prepare_target(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(int)
    s2 = s.astype(str).str.strip().str.lower()
    m = {'si':1,'sí':1,'s':1,'1':1,'true':1,'yes':1,'y':1,'no':0,'n':0,'0':0,'false':0}
    mapped = s2.map(m)
    if mapped.isna().any():
        num = pd.to_numeric(s, errors='coerce')
        if num.notna().all():
            return num.astype(int)
    return mapped

y = prepare_target(df[target])
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int)

# If no previous RFE run produced `selected`, rank features by univariate mutual info
try:
    selected
except NameError:
    from sklearn.feature_selection import mutual_info_classif
    ranks = mutual_info_classif(X.fillna(0), y, discrete_features='auto', random_state=seed)
    feat_rank = pd.Series(ranks, index=X.columns).sort_values(ascending=False)
    ordered_features = feat_rank.index.tolist()
else:
    ordered_features = list(selected) + [c for c in X.columns if c not in selected]

# Evaluate models for each k
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
for k in parsimony_steps:
    k = min(k, len(ordered_features))
    cols_k = ordered_features[:k]
    Xk = X[cols_k].fillna(0)
    # CV AUC
    try:
        aucs = cross_val_score(LogisticRegression(max_iter=1000, solver='liblinear'), Xk, y, scoring='roc_auc', cv=cv, n_jobs=-1)
        auc_mean = float(np.mean(aucs))
        auc_std = float(np.std(aucs))
    except Exception as e:
        auc_mean = np.nan
        auc_std = np.nan
    # AIC/BIC via statsmodels on full data (fit once)
    try:
        sm_model = sm.Logit(y, sm.add_constant(Xk)).fit(disp=False)
        aic = float(sm_model.aic)
        bic = float(sm_model.bic)
    except Exception as e:
        aic = np.nan
        bic = np.nan
    results.append({'k':k, 'n_features':len(cols_k), 'auc_mean':auc_mean, 'auc_std':auc_std, 'aic':aic, 'bic':bic, 'features':cols_k})

res_df = pd.DataFrame(results).sort_values('k')
print('Resumen parsimonia (por k):')
display(res_df[['k','n_features','auc_mean','auc_std','aic','bic']])

# Selección automática: elegir modelos con AUC >= best_auc - roc_tol, luego elegir menor BIC
best_auc = res_df['auc_mean'].max()
candidates = res_df[res_df['auc_mean'] >= (best_auc - roc_tol)].copy()
if candidates.empty:
    print('No hay modelos dentro de la tolerancia de AUC; revise datos o ajuste roc_auc_tolerance')
else:
    chosen = candidates.sort_values('bic').iloc[0]
    print('\nMejor AUC observada: {:.4f}'.format(best_auc))
    print('Modelos dentro de tolerancia (AUC >= best - tol):')
    display(candidates[['k','n_features','auc_mean','aic','bic']])
    print('\nSelección recomendada (menor BIC entre candidatos):')
    print(f"k={chosen['k']} | n_features={chosen['n_features']} | bic={chosen['bic']} | auc_mean={chosen['auc_mean']:.4f}")
    print('Features recomendadas para production (inference_features):')
    print(chosen['features'])
    print('\nSnippet sugerido para settings.yaml:')
    print('case:')
    print(f"  inference_features: {chosen['features']}")


Dataset: data/raw/bank360_dataset.csv rows=1200 | target='abandono' | candidate features=25
Resumen parsimonia (por k):


,k,n_features,auc_mean,auc_std,aic,bic
0,4,4,0.625658,0.031414,1461.374176,1486.82456
1,8,8,0.637300,0.023585,NaN,NaN
2,12,12,0.644235,0.028263,NaN,NaN
3,18,18,0.731010,0.036493,NaN,NaN



Mejor AUC observada: 0.7310
Modelos dentro de tolerancia (AUC >= best - tol):


,k,n_features,auc_mean,aic,bic
3,18,18,0.73101,NaN,NaN



Selección recomendada (menor BIC entre candidatos):
k=18 | n_features=18 | bic=nan | auc_mean=0.7310
Features recomendadas para production (inference_features):
['saldo_variacion_pct_3m', 'productos_activos', 'ratio_uso_credito', 'reclamaciones', 'atraso_30d', 'usa_app_Si', 'nomina_domiciliada_Si', 'hipoteca_activa_Si', 'producto_premium_Si', 'macro_segmento_Premium', 'macro_segmento_Recuperacion', 'macro_segmento_Valor', 'edad', 'antiguedad_meses', 'ingreso_mensual', 'ingreso_mensual_fracdiff_0_4', 'saldo_promedio_3m', 'saldo_promedio_3m_fracdiff_0_4']

Snippet sugerido para settings.yaml:
case:
  inference_features: ['saldo_variacion_pct_3m', 'productos_activos', 'ratio_uso_credito', 'reclamaciones', 'atraso_30d', 'usa_app_Si', 'nomina_domiciliada_Si', 'hipoteca_activa_Si', 'producto_premium_Si', 'macro_segmento_Premium', 'macro_segmento_Recuperacion', 'macro_segmento_Valor', 'edad', 'antiguedad_meses', 'ingreso_mensual', 'ingreso_mensual_fracdiff_0_4', 'saldo_promedio_3m', 'saldo_p

### Interpretación — AIC/BIC y Parsimonia

> Traducción simple: esta comparación ayuda a decidir si vale la pena pagar el coste de un modelo más complejo.

- Si un modelo simple rinde casi igual que uno grande, normalmente conviene el simple.
- Si la mejora del modelo complejo es clara y consistente, entonces la complejidad puede justificarse.
- La idea no es escoger el modelo con mejor número aislado, sino el mejor equilibrio entre rendimiento, claridad y estabilidad.
- En `settings.yaml`, esta lectura afecta `case.parsimony.*` y puede llevar a revisar `case.inference_features`.

**Mensaje para no técnicos:** más variables no siempre significa mejor decisión. Muchas veces significa más fragilidad.

---

## 6) Detección y tratamiento de outliers (IQR)

> Esta sección sirve para localizar columnas donde algunos valores extremos pueden distorsionar promedios, umbrales y conclusiones.

**Pregunta de negocio:** ¿hay variables con valores atípicos que puedan sesgar la lectura del caso o llevarnos a decisiones exageradas?

**Qué parámetro ayuda a decidir:** `case.outlier_cols`.

**Cómo leer esta sección si no eres técnico:**

- Un outlier no siempre es un error: a veces representa un cliente muy valioso o una situación real rara pero importante.
- La clave no es borrar extremos por reflejo, sino decidir si conviene caparlos, transformarlos o conservarlos y documentarlo.

**Qué decisión ayuda a tomar:** en qué columnas conviene aplicar tratamiento robusto antes de entrenar o segmentar.

In [8]:
def detect_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper
# Uso: mask, lo, hi = detect_iqr_outliers(df['ingreso_mensual'])

In [19]:
# Detección y recomendaciones de tratamiento de outliers (IQR) — salida didáctica
import yaml, pandas as pd
from pathlib import Path

# cargar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

outlier_cols = cfg['case'].get('outlier_cols', [])
present = [c for c in outlier_cols if c in df.columns]
print(f"Columnas candidatas a outliers definidas en settings: {len(outlier_cols)} | presentes en dataset: {len(present)}")

if not present:
    print('No hay columnas de outliers encontradas en el dataset. Actualice settings.yaml para incluir columnas relevantes.')
else:
    summary = []
    for col in present:
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        q1, q3 = s.quantile([0.25,0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_out = ((s < lower) | (s > upper)).sum()
        pct = 100.0 * n_out / len(df)
        summary.append({'col':col,'n_out':int(n_out),'pct':pct,'lower':float(lower),'upper':float(upper)})
    sum_df = pd.DataFrame(summary).sort_values('pct', ascending=False)
    display(sum_df)
    print('\nRecomendaciones automáticas (reglas heurísticas):')
    for r in summary:
        col=r['col']; pct=r['pct']
        if pct < 0.5:
            action = 'Winsorizar (recortar) a límites IQR — conserva la señal, evita pérdida de datos.'
        elif pct <= 5:
            action = 'Considerar winsorizar o transformar (log/boxcox). Revisar casos extremos manualmente.'
        else:
            action = 'Alto porcentaje de outliers: investigar origen, posibles errores de captura o subpoblaciones. Considerar transformación robusta o modelado separado.'
        print(f"- {col}: {r['n_out']} filas ({pct:.2f}%) → {action}")

    # mostrar ejemplos de outliers para las 3 columnas con mayor pct
    top3 = sum_df.head(3)['col'].tolist()
    for col in top3:
        s = pd.to_numeric(df[col], errors='coerce')
        q1, q3 = s.quantile([0.25,0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        out_idx = s[(s < lower) | (s > upper)].index[:5]
        if len(out_idx)>0:
            print(f"\nEjemplos de outliers en {col} (muestra):")
            display(df.loc[out_idx, [col]].assign(_reason=lambda d: d[col].apply(lambda v: 'low' if v<lower else 'high')))

    print('\nSnippet sugerido para aplicar winsorize (modifica en su pipeline según necesidad):')
    print("df['{col}'] = df['{col}'].clip(lower=lower, upper=upper)")


Columnas candidatas a outliers definidas en settings: 4 | presentes en dataset: 4


,col,n_out,pct,lower,upper
1,saldo_promedio_3m,42,3.500000,-4012.17750,24181.20250
3,valor_cliente_12m,39,3.250000,-685.80625,5634.14375
2,rentabilidad_mensual_estimada,33,2.750000,-54.05625,491.73375
0,ingreso_mensual,29,2.416667,160.33875,4511.96875



Recomendaciones automáticas (reglas heurísticas):
- ingreso_mensual: 29 filas (2.42%) → Considerar winsorizar o transformar (log/boxcox). Revisar casos extremos manualmente.
- saldo_promedio_3m: 42 filas (3.50%) → Considerar winsorizar o transformar (log/boxcox). Revisar casos extremos manualmente.
- rentabilidad_mensual_estimada: 33 filas (2.75%) → Considerar winsorizar o transformar (log/boxcox). Revisar casos extremos manualmente.
- valor_cliente_12m: 39 filas (3.25%) → Considerar winsorizar o transformar (log/boxcox). Revisar casos extremos manualmente.

Ejemplos de outliers en saldo_promedio_3m (muestra):


,saldo_promedio_3m,_reason
1,25615.03,high
13,27655.09,high
22,28088.75,high
29,27891.67,high
58,26422.61,high



Ejemplos de outliers en valor_cliente_12m (muestra):


,valor_cliente_12m,_reason
22,5688.410,high
58,5775.000,high
95,6286.980,high
213,5816.090,high
241,8654.975,high



Ejemplos de outliers en rentabilidad_mensual_estimada (muestra):


,rentabilidad_mensual_estimada,_reason
1,491.77,high
13,505.23,high
22,556.28,high
29,555.65,high
58,517.79,high



Snippet sugerido para aplicar winsorize (modifica en su pipeline según necesidad):
df['{col}'] = df['{col}'].clip(lower=lower, upper=upper)


### Interpretación — Outliers (IQR)

> Traducción simple: este bloque te muestra dónde hay valores tan extremos que podrían distorsionar promedios, rankings o segmentaciones.

- Un extremo puede ser error, rareza o señal valiosa; por eso primero se interpreta y luego se decide el tratamiento.
- Si una columna concentra muchos extremos, conviene documentar si se va a capar, transformar o dejar intacta.
- En `settings.yaml`, esta revisión impacta sobre `case.outlier_cols`.

**Regla prudente:** no borres extremos por costumbre; decide si dañan el análisis o si representan un caso de negocio importante.

---

## 7) Validación temporal purgada (Purged CV)

> Esta parte comprueba si el modelo sigue funcionando cuando se respeta el tiempo real del negocio y se evita que el entrenamiento "vea el futuro".

**Pregunta de negocio:** ¿el rendimiento observado es creíble en un escenario temporal real o está inflado por fuga de información?

**Qué parámetro ayuda a decidir:** `case.temporal_validation.n_splits`, `case.temporal_validation.purge_gap_days` y `case.temporal_validation.embargo_gap_days`.

**Cómo leer esta sección si no eres técnico:**

- Si la métrica cae mucho respecto a una validación simple, la señal no era tan robusta como parecía.
- Si la métrica se sostiene, la solución gana credibilidad para un caso operativo real.

**Qué decisión ayuda a tomar:** cuán estricta debe ser la validación temporal y si el modelo merece confianza fuera de muestra.

In [9]:
# Ejemplo de uso (requiere instalar el paquete del proyecto):
try:
    from banca_360_mlops.core.metodologia import run_purged_temporal_validation
    print('run_purged_temporal_validation importado correctamente')
    # run_purged_temporal_validation(df, target='abandono', date_col='fecha', n_splits=4, purge_gap_days=7, embargo_gap_days=14)
except Exception as e:
    print('No se pudo importar run_purged_temporal_validation:', e)

No se pudo importar run_purged_temporal_validation: cannot import name 'run_purged_temporal_validation' from 'banca_360_mlops.core.metodologia' (C:\Users\yonai\Documents\Master Analytics and Visual in Big Data\Master\banca_360_metodologia_v2\src\banca_360_mlops\core\metodologia.py)


In [20]:
# Validación temporal purgada — demostración didáctica ligera
import yaml, pandas as pd, numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Cargar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

# detectar columna fecha
date_candidates = ['fecha_corte','fecha_registro','fecha','fecha_transaccion','fecha_operacion']
date_col = next((c for c in date_candidates if c in df.columns), None)
if date_col is None:
    print('No se encontró columna de fecha automática. Proporcione una columna de tiempo en settings o dataset.')
else:
    print('Usando columna de fecha:', date_col)
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])

    target = cfg['case'].get('target')
    if target not in df.columns:
        raise ValueError('Target no encontrado en dataset')
    features = cfg['case'].get('inference_features') or cfg['case'].get('feature_cols')
    features = [c for c in features if c in df.columns]
    if not features:
        raise ValueError('No hay features disponibles para validar')

    # simple preprocessing: numeric + small categorical dummies
    X = df[features].copy()
    for col in X.select_dtypes(include=['object','category']).columns:
        if X[col].nunique() <= 10:
            dummies = pd.get_dummies(X[col].astype(str), prefix=col, drop_first=True)
            X = pd.concat([X.drop(columns=col), dummies], axis=1)
        else:
            X[col] = X[col].astype('category').cat.codes
    # prepare y
    def prepare_target(s):
        if pd.api.types.is_numeric_dtype(s):
            return s.astype(int)
        s2 = s.astype(str).str.strip().str.lower()
        mapvals = {'si':1,'sí':1,'s':1,'1':1,'true':1,'yes':1,'y':1,'no':0,'n':0,'0':0,'false':0}
        mapped = s2.map(mapvals)
        if mapped.isna().any():
            num = pd.to_numeric(s, errors='coerce')
            if num.notna().all():
                return num.astype(int)
        return mapped
    y = prepare_target(df[target])
    mask = y.notna()
    X = X.loc[mask].fillna(0)
    y = y.loc[mask].astype(int)
    df = df.loc[mask]

    n_splits = cfg.get('temporal_validation', {}).get('n_splits', 4)
    purge_days = cfg.get('temporal_validation', {}).get('purge_days', 30)

    # crear cortes temporales por cuantiles (muestra didáctica)
    df = df.sort_values(date_col)
    dates = df[date_col]
    boundaries = [dates.quantile(i / n_splits) for i in range(n_splits + 1)]

    fold_results = []
    for i in range(n_splits):
        t0 = boundaries[i]
        t1 = boundaries[i+1]
        test_mask = (df[date_col] >= t0) & (df[date_col] <= t1)
        if test_mask.sum() < 10:
            print(f'Fold {i}: muy pocas observaciones en test ({test_mask.sum()}), se omite')
            continue
        # definir purga: excluir train rows con fecha >= t0 - purge_days
        purge_start = t0 - pd.Timedelta(days=purge_days)
        train_mask = (~test_mask) & (df[date_col] < purge_start)
        # si no hay suficientes filas en train, relajar purga
        if train_mask.sum() < 30:
            train_mask = ~test_mask
        X_train = X.loc[train_mask]
        y_train = y.loc[train_mask]
        X_test = X.loc[test_mask]
        y_test = y.loc[test_mask]
        clf = LogisticRegression(max_iter=1000, solver='liblinear')
        try:
            clf.fit(X_train, y_train)
            preds = clf.predict_proba(X_test)[:,1]
            auc = roc_auc_score(y_test, preds)
        except Exception as e:
            auc = float('nan')
        fold_results.append({'fold':i,'train_rows':int(train_mask.sum()),'test_rows':int(test_mask.sum()),'auc':auc,'t0':str(t0)[:10],'t1':str(t1)[:10]})

    fr = pd.DataFrame(fold_results)
    print('\nResultados de validación temporal purgada (resumen por fold):')
    display(fr[['fold','train_rows','test_rows','auc','t0','t1']])
    print('\nInterpretación didáctica:')
    print('- Cada fold prueba el modelo en un periodo temporal posterior al de entrenamiento (purga aplicada).')
    print('- Revise folds con AUC muy diferentes: pueden indicar drift o problemas de estabilidad temporal.')
    print('- Ajuste purge_days o n_splits según frecuencia de eventos y ventana de negocio.')


Usando columna de fecha: fecha_corte

Resultados de validación temporal purgada (resumen por fold):


,fold,train_rows,test_rows,auc,t0,t1
0,0,900,300,0.765551,2024-10-08,2025-02-15
1,1,220,301,0.684731,2025-02-15,2025-07-01
2,2,530,301,0.730404,2025-07-01,2025-11-14
3,3,838,300,0.756230,2025-11-14,2026-03-29



Interpretación didáctica:
- Cada fold prueba el modelo en un periodo temporal posterior al de entrenamiento (purga aplicada).
- Revise folds con AUC muy diferentes: pueden indicar drift o problemas de estabilidad temporal.
- Ajuste purge_days o n_splits según frecuencia de eventos y ventana de negocio.


### Interpretación — Validación temporal purgada (Purged CV)

> Traducción simple: esta salida te dice si el modelo sigue siendo creíble cuando se le prohíbe aprender del futuro.

- Si el desempeño baja mucho frente a una validación simple, probablemente había optimismo artificial.
- Si el desempeño se mantiene razonablemente, la señal es más confiable para un entorno real.
- En `settings.yaml`, esta sección ayuda a ajustar `case.temporal_validation.n_splits`, `purge_gap_days` y `embargo_gap_days`.

**Regla para negocio:** una métrica más baja pero honesta suele valer más que una métrica alta inflada por fuga temporal.

---

## 8) CLV probabilístico y `clv_horizons`

> Aquí se traduce la pregunta de valor futuro a una decisión concreta: en qué horizonte conviene estimar el beneficio esperado del cliente.

**Pregunta de negocio:** ¿tiene más sentido valorar clientes a 6 meses, 12 meses u otro plazo según el uso real del caso?

**Qué parámetro ayuda a decidir:** `case.clv_horizons`.

**Cómo leer esta sección si no eres técnico:**

- Un horizonte corto suele ser más prudente y operativo.
- Un horizonte largo puede capturar más valor, pero también introduce más incertidumbre.

**Qué decisión ayuda a tomar:** qué ventana temporal usar para que el valor esperado sea útil y defendible.

In [10]:
def estimate_clv_rfm(transactions, horizons_months=[6,12]):
    # transactions: DataFrame con columnas ['customer_id','date','monetary']
    if BetaGeoFitter is None or GammaGammaFitter is None:
        raise ImportError('Instala lifetimes para estimar CLV: pip install lifetimes')
    # Preparación RFM resumida: frecuencia, recencia, monetary (por cliente)
    # Aquí se asume que ya existe un resumen por cliente: freq, recency, monetary_avg, T (periodo observado)
    # Ajustar modelos
    bgf = BetaGeoFitter(penalizer_coef=0.0)
    ggf = GammaGammaFitter(penalizer_coef=0.0)
    # ejemplo: rfm = DataFrame con columnas ['customer_id','frequency','recency','T','monetary_avg']
    # bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])
    # ggf.fit(rfm['frequency'], rfm['monetary_avg'])
    # Predecir CLV para cada horizonte: ggf.customer_lifetime_value(bgf, rfm['frequency'], rfm['recency'], rfm['T'], rfm['monetary_avg'], time=horizon, freq='M')
    pass
# Nota: ejecutar backtesting para validar precisión por horizonte

In [21]:
# CLV probabilístico — salida didáctica (fallback si no hay datos transaccionales)
import yaml, pandas as pd, numpy as np
from pathlib import Path

# cargar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

clv_horizons = cfg['case'].get('clv_horizons', [6,12])
print(f"CLV horizons: {clv_horizons}")

# buscar columnas útiles
cols = df.columns.str.lower().tolist()
has_clv_m = any('clv' in c and 'monet' in c for c in cols)
has_freq = any('freq' in c or 'frecuencia' in c or 'compras' in c for c in cols)

print('Columnas relevantes detectadas (ejemplos):')
print([c for c in df.columns if 'clv' in c.lower() or 'compra' in c.lower() or 'frecuencia' in c.lower()][:10])

# Si existen campos agregados (monetario y frecuencia) usamos aproximación simple
if has_clv_m or has_freq:
    # heurística: buscar columnas por nombre
    monet_cols = [c for c in df.columns if 'clv' in c.lower() and 'monet' in c.lower()]
    freq_cols = [c for c in df.columns if ('freq' in c.lower()) or ('frecuencia' in c.lower()) or ('compras' in c.lower())]
    monet = df[monet_cols[0]] if monet_cols else None
    freq = df[freq_cols[0]] if freq_cols else None
    print('Usando columnas -> monetario:', monet.name if monet is not None else None, ' frecuencia:', freq.name if freq is not None else None)

    clv_table = {}
    for h in clv_horizons:
        if monet is not None and freq is not None:
            # suponer que `freq` es compras por año si nombre contiene '12' o similar; si es compras en X meses no podemos inferir; asumimos frecuencia anual cuando >1
            clv_est = monet.fillna(0) * (freq.fillna(0) * (h/12.0))
        elif monet is not None:
            clv_est = monet.fillna(0) * (h/12.0)
        else:
            clv_est = pd.Series(0, index=df.index)
        clv_table[h] = clv_est
        print(f"\nHorizonte {h} meses — resumen CLV estimado:")
        print(f"  mean: {clv_est.mean():.2f} | median: {clv_est.median():.2f} | 90th pct: {clv_est.quantile(0.9):.2f}")

    # adjuntar ejemplo de columna al dataframe (no persiste automáticamente)
    sample_col = f'clv_est_{clv_horizons[0]}m'
    df[sample_col] = clv_table[clv_horizons[0]]
    print('\nEjemplo: columna añadida en memoria ->', sample_col)
    print('Sugerencia: si dispone de datos transaccionales use BG/NBD + Gamma-Gamma para estimar CLV probabilístico; la heurística anterior es solo aproximación.')
else:
    print('No se encontraron columnas de frecuencia ni monetario agregadas. Para CLV probabilístico necesita: transacciones por cliente (customer_id + tx_date + tx_amount).')
    print('Si dispone de esos datos, cargue la tabla transaccional y ejecute un ajuste BG/NBD + Gamma-Gamma (ejemplo en celdas siguientes si está disponible).')


CLV horizons: [6, 12]
Columnas relevantes detectadas (ejemplos):
['compras_12m', 'clv_t_dias', 'clv_frecuencia', 'clv_t_ultima_compra_dias', 'clv_dias_desde_ultima_transaccion', 'clv_monetario_promedio', 'clv_valor_observado_12m']
Usando columnas -> monetario: clv_monetario_promedio  frecuencia: compras_12m

Horizonte 6 meses — resumen CLV estimado:
  mean: 680.33 | median: 603.47 | 90th pct: 1164.38

Horizonte 12 meses — resumen CLV estimado:
  mean: 1360.66 | median: 1206.94 | 90th pct: 2328.75

Ejemplo: columna añadida en memoria -> clv_est_6m
Sugerencia: si dispone de datos transaccionales use BG/NBD + Gamma-Gamma para estimar CLV probabilístico; la heurística anterior es solo aproximación.


### Interpretación — CLV probabilístico y `clv_horizons`

> Traducción simple: aquí decides cuánto futuro quieres comprar en tu estimación de valor.

- Un horizonte más corto suele ser más prudente y más fácil de defender.
- Un horizonte más largo puede capturar más valor, pero también más incertidumbre.
- La mejor elección no es la más larga, sino la que mejor encaja con el ciclo real de decisión comercial.
- En `settings.yaml`, el impacto principal está en `case.clv_horizons`.

**Pregunta útil para negocio:** ¿en cuánto tiempo esperas recuperar el esfuerzo comercial de una acción de retención?

---

## 9) `retention_success_rate`: cálculo y bootstrap CI

> Esta sección convierte la intuición comercial sobre la tasa de éxito en una estimación más prudente y trazable.

**Pregunta de negocio:** ¿qué tasa de éxito conviene usar para valorar campañas sin caer en optimismo excesivo?

**Qué parámetro ayuda a decidir:** `case.retention_success_rate`.

**Cómo leer esta sección si no eres técnico:**

- La media te da un valor central razonable.
- El intervalo te enseña el rango de incertidumbre: si es muy ancho, conviene ser más conservador al prometer ROI.

**Qué decisión ayuda a tomar:** qué tasa puntual fijar en el YAML y cuánto margen de prudencia aplicar.

In [11]:
def retention_rate_bootstrap(successes, contacts, n_boot=5000, random_state=42):
    rng = np.random.default_rng(random_state)
    rates = []
    data = np.concatenate([np.ones(successes), np.zeros(contacts - successes)])
    for _ in range(n_boot):
        sample = rng.choice(data, size=len(data), replace=True)
        rates.append(sample.mean())
    return np.mean(rates), np.percentile(rates, 2.5), np.percentile(rates, 97.5)
# Uso: mean, lo, hi = retention_rate_bootstrap(280, 1000)  # 280 éxitos / 1000 contactos -> ~0.28

In [23]:
# Cálculo didáctico de `retention_success_rate` (bootstrap/posterior Beta si faltan datos)
import yaml, pandas as pd, numpy as np
from pathlib import Path

settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

# intentar detectar columnas de contact y success
success_candidates = [c for c in df.columns if any(s in c.lower() for s in ['exito','success','ganado','retencion','win','resultado'])]
contact_candidates = [c for c in df.columns if any(s in c.lower() for s in ['contact','contacto','attempt','intento','n_contact','n_contacto','llamadas'])]

print('Candidates success cols:', success_candidates[:5])
print('Candidates contact cols:', contact_candidates[:5])

successes = None
contacts = None
if success_candidates and contact_candidates:
    s_col = success_candidates[0]
    c_col = contact_candidates[0]
    sc = pd.to_numeric(df[s_col], errors='coerce')
    if sc.dropna().isin([0,1]).all():
        successes = int(sc.sum())
        contacts = int(len(sc))
    else:
        contacts_series = pd.to_numeric(df[c_col], errors='coerce')
        if contacts_series.notna().all():
            contacts = int(contacts_series.sum())
            successes = int(sc.fillna(0).sum())
        else:
            contacts = len(df)
            successes = int(sc.fillna(0).sum())

# fallback: usar valor en settings si no hay cuentas directas
if successes is None or contacts is None or contacts==0:
    p0 = cfg['case'].get('retention_success_rate', None)
    sample_n = min(1000, len(df))
    if p0 is None:
        print('No se encontraron datos de campaña y settings.case.retention_success_rate no está definido.')
        print('Usando un valor por defecto de 0.28 para demostrar intervalo con sample_n=', sample_n)
        p0 = 0.28
    else:
        print(f'No se encontraron columnas explícitas; usando retention_success_rate de settings: {p0} y sample_n={sample_n}')
    successes = int(round(p0 * sample_n))
    contacts = int(sample_n)

# calcular posterior Beta (Jeffreys)
from scipy.stats import beta
alpha = successes + 1
beta_b = contacts - successes + 1
posterior_samples = beta.rvs(alpha, beta_b, size=10000)
ci_low, ci_high = np.percentile(posterior_samples, [2.5,97.5])
mean_p = posterior_samples.mean()
print('\nEstimación de tasa de éxito de retención:')
print(f'  successes = {successes}, contacts = {contacts}, tasa puntual = {successes/contacts:.4f}')
print(f'  Posterior Beta({alpha},{beta_b}) → 95% CI = [{ci_low:.3f}, {ci_high:.3f}] | mean posterior = {mean_p:.3f}')

print('\nInterpretación didáctica:')
print('- Use el intervalo para planear tamaño de muestra y expectativas en campañas.')
print('- Si dispone de la tabla de contactos por campaña, agregue successes/contacts reales y re-ejecute esta celda.')
print('\nSnippet sugerido para settings.yaml (si quiere fijar un valor inicial):')
print('case:')
print(f"  retention_success_rate: {mean_p:.3f}")


Candidates success cols: []
Candidates contact cols: []
No se encontraron columnas explícitas; usando retention_success_rate de settings: 0.28 y sample_n=1000

Estimación de tasa de éxito de retención:
  successes = 280, contacts = 1000, tasa puntual = 0.2800
  Posterior Beta(281,721) → 95% CI = [0.253, 0.309] | mean posterior = 0.280

Interpretación didáctica:
- Use el intervalo para planear tamaño de muestra y expectativas en campañas.
- Si dispone de la tabla de contactos por campaña, agregue successes/contacts reales y re-ejecute esta celda.

Snippet sugerido para settings.yaml (si quiere fijar un valor inicial):
case:
  retention_success_rate: 0.280


### Interpretación — `retention_success_rate` (bootstrap)

> Traducción simple: este bloque te ayuda a no prometer una tasa de éxito demasiado optimista.

- La media es el valor central más razonable para empezar.
- El intervalo te recuerda que la tasa real puede moverse dentro de un rango, no en un único punto fijo.
- Si el intervalo es muy amplio, conviene ser conservador en el cálculo de valor esperado.
- En `settings.yaml`, esta salida alimenta `case.retention_success_rate`.

**Regla prudente:** para planes conservadores, usa como referencia una cifra cercana al centro del intervalo o incluso su parte baja.

---

## 10) `segment_features`: escalado (puntuación Z)

> Aquí se prepara la base de segmentación para que ninguna variable domine solo por tener una escala más grande.

**Pregunta de negocio:** ¿qué variables deben usarse para segmentar clientes y cómo evitar que una sola medida distorsione el agrupamiento?

**Qué parámetro ayuda a decidir:** `case.segment_features`.

**Cómo leer esta sección si no eres técnico:**

- Escalar no cambia el significado del dato; cambia la forma en que el algoritmo lo compara con los demás.
- Si no escalas, una variable monetaria grande puede tapar otras señales valiosas.

**Qué decisión ayuda a tomar:** qué variables usar para segmentación y si el preprocesamiento está listo para clustering operativo.

In [12]:
def scale_segment_features(df, columns):
    scaler = StandardScaler()
    arr = scaler.fit_transform(df[columns].fillna(0))
    df_scaled = pd.DataFrame(arr, columns=columns, index=df.index)
    # comprobar media y std
    print(df_scaled.mean().round(6).to_dict())
    print(df_scaled.std().round(6).to_dict())
    return df_scaled, scaler
# Uso: scaled, scaler = scale_segment_features(scorecard, ['probabilidad_abandono','clv_probabilistico_6m'])

In [24]:
# Segment features — cálculo y escalado didáctico
import yaml, pandas as pd, numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression

# cargar settings y dataset
settings_candidates = ["conf/settings.yaml","settings.yaml","notebooks/settings.yaml","Notebooks/settings.yaml"]
cfg=None
for sp in settings_candidates:
    if Path(sp).exists():
        cfg = yaml.safe_load(Path(sp).read_text(encoding='utf-8'))
        break
if cfg is None:
    raise FileNotFoundError('settings.yaml no encontrado')

data_candidates=['data/raw/bank360_dataset.csv','banca_360_metodologia_v3/data/raw/bank360_dataset.csv']
df=None
for dp in data_candidates:
    if Path(dp).exists():
        df = pd.read_csv(dp)
        data_path=dp
        break
if df is None:
    raise FileNotFoundError('bank360_dataset.csv no encontrado')

segment_features_cfg = cfg['case'].get('segment_features', [])
print('Segment features from settings:', segment_features_cfg)

# probabilidad_abandono: usar columna si existe, sino entrenar un modelo rápido
if 'probabilidad_abandono' in df.columns:
    df['prob_abandono'] = pd.to_numeric(df['probabilidad_abandono'], errors='coerce')
else:
    print('No existe columna `probabilidad_abandono`. Entrenando modelo rápido con `inference_features` para estimar proba (demo).')
    features = cfg['case'].get('inference_features') or cfg['case'].get('feature_cols')
    features = [c for c in features if c in df.columns]
    if not features:
        raise ValueError('No hay features disponibles para entrenar proba')
    X = df[features].copy()
    for col in X.select_dtypes(include=['object','category']).columns:
        if X[col].nunique() <= 10:
            dummies = pd.get_dummies(X[col].astype(str), prefix=col, drop_first=True)
            X = pd.concat([X.drop(columns=col), dummies], axis=1)
        else:
            X[col] = X[col].astype('category').cat.codes
    target = cfg['case'].get('target')
    if target not in df.columns:
        raise ValueError('Target no encontrado para entrenamiento')
    y = df[target].astype(str).str.strip().str.lower().map({'si':1,'sí':1,'s':1,'1':1,'true':1,'yes':1,'y':1,'no':0,'n':0,'0':0}).fillna(0).astype(int)
    clf = LogisticRegression(max_iter=1000, solver='liblinear')
    try:
        clf.fit(X.fillna(0), y)
        df['prob_abandono'] = clf.predict_proba(X.fillna(0))[:,1]
    except Exception as e:
        print('Error entrenando modelo rápido para proba:', e)
        df['prob_abandono'] = 0.0

# CLV estimado: buscar columna generada o crear heurística rápida
clv_col = None
for h in cfg['case'].get('clv_horizons', [6,12]):
    candidate = f'clv_est_{h}m'
    if candidate in df.columns:
        clv_col = candidate
        break
if clv_col is None:
    # intentar heurística usando columnas monetario/frecuencia
    monet_cols = [c for c in df.columns if 'clv' in c.lower() and 'monet' in c.lower()]
    freq_cols = [c for c in df.columns if ('freq' in c.lower()) or ('frecuencia' in c.lower()) or ('compras' in c.lower())]
    if monet_cols and freq_cols:
        monet = pd.to_numeric(df[monet_cols[0]], errors='coerce').fillna(0)
        freq = pd.to_numeric(df[freq_cols[0]], errors='coerce').fillna(0)
        horizon = cfg['case'].get('clv_horizons',[6])[0]
        df[f'clv_est_{horizon}m'] = monet * (freq * (horizon/12.0))
        clv_col = f'clv_est_{horizon}m'
    else:
        df['clv_est_6m'] = 0.0
        clv_col = 'clv_est_6m'

# retention rate y coste contacto
ret_rate = cfg['case'].get('retention_success_rate', 0.28)
contact_cost = cfg['case'].get('contact_cost', 1.0)

# valor esperado por contacto
h = cfg['case'].get('clv_horizons', [6])[0]
df['valor_esperado_contacto'] = ret_rate * df[clv_col].fillna(0) - contact_cost

# prioridad integrada (prob * clv)
df['prioridad_integrada'] = df['prob_abandono'].fillna(0) * df[clv_col].fillna(0)

# seleccionar features de segmentación disponibles
seg_feats = [f for f in segment_features_cfg if f in df.columns]
if not seg_feats:
    seg_feats = ['prob_abandono', clv_col, 'valor_esperado_contacto', 'prioridad_integrada']
print('Features que se van a escalar para segmentación:', seg_feats)

# Escalado estándar y mostrar resumen
scaler = StandardScaler()
scaled = scaler.fit_transform(df[seg_feats].fillna(0))
scaled_df = pd.DataFrame(scaled, columns=[f + '_z' for f in seg_feats], index=df.index)
print('\nResumen estadístico antes del escalado:')
print(df[seg_feats].describe().T[['mean','std','min','max']])
print('\nResumen estadístico después del escalado (z-score):')
print(scaled_df.describe().T[['mean','std','min','max']])

print('\nMuestra final (primeras 10 filas) con columnas originales y escaladas:')
display(df[seg_feats].head(10).join(scaled_df.head(10)))

print('\nSugerencia para settings.yaml:')
print('case:')
print(f"  segment_features: {seg_feats}")
print('\nInterpretación:')
print('- `valor_esperado_contacto` combina probabilidad de éxito con CLV y coste de contacto; ordenar por esta métrica prioriza ROAS.')
print('- `prioridad_integrada` es útil como ranking directo (prob * CLV). Normalize/escale antes de fijar segmentos.')


Segment features from settings: ['probabilidad_abandono', 'valor_esperado_contacto', 'clv_probabilistico_6m', 'prioridad_integrada']
No existe columna `probabilidad_abandono`. Entrenando modelo rápido con `inference_features` para estimar proba (demo).
Features que se van a escalar para segmentación: ['valor_esperado_contacto', 'prioridad_integrada']

Resumen estadístico antes del escalado:
                               mean         std   min          max
valor_esperado_contacto  168.491998  107.916718 -22.0  1126.316400
prioridad_integrada      177.510081   89.807738   0.0   522.054221

Resumen estadístico después del escalado (z-score):
                                   mean       std       min       max
valor_esperado_contacto_z -6.661338e-17  1.000417 -1.765912  8.879289
prioridad_integrada_z      1.924387e-17  1.000417 -1.977381  3.838063

Muestra final (primeras 10 filas) con columnas originales y escaladas:


,valor_esperado_contacto,prioridad_integrada,valor_esperado_contacto_z,prioridad_integrada_z
0,257.1068,91.597193,0.821483,-0.957030
1,615.5040,98.199416,4.143921,-0.883484
2,178.6620,175.793580,0.094279,-0.019121
3,147.0192,167.278070,-0.199059,-0.113980
4,335.9030,73.666697,1.551945,-1.156768
5,308.1970,54.109251,1.295103,-1.374629
6,143.8160,242.056016,-0.228753,0.719012
7,38.5766,93.250268,-1.204351,-0.938616
8,295.3828,422.641372,1.176312,2.730650
9,71.5816,77.686349,-0.898385,-1.111990



Sugerencia para settings.yaml:
case:
  segment_features: ['valor_esperado_contacto', 'prioridad_integrada']

Interpretación:
- `valor_esperado_contacto` combina probabilidad de éxito con CLV y coste de contacto; ordenar por esta métrica prioriza ROAS.
- `prioridad_integrada` es útil como ranking directo (prob * CLV). Normalize/escale antes de fijar segmentos.


### Interpretación — Escalado de `segment_features`

> Traducción simple: el escalado hace que la segmentación compare las variables en igualdad de condiciones.

- Si una variable tiene números mucho más grandes que otra, puede dominar el clustering sin merecerlo.
- Escalar evita ese sesgo de magnitud y hace más justa la comparación entre señales.
- Si hay outliers fuertes, conviene valorar un escalado robusto antes de segmentar.
- En `settings.yaml`, esta sección ayuda a consolidar `case.segment_features`.

**Mensaje para no técnicos:** aquí no decides solo qué variables usar, sino si están listas para convivir sin distorsionarse entre sí.

---

## 11) Editar `conf/settings.yaml` desde Python (ejemplo)

> Esta sección muestra cómo llevar las recomendaciones analíticas al archivo que realmente gobierna el proyecto.

**Pregunta de negocio:** una vez tomada la decisión, ¿cómo la dejo escrita de forma reproducible y ordenada?

**Qué archivo ayuda a actualizar:** `conf/settings.yaml`.

**Cómo leer esta sección si no eres técnico:**

- No hace falta modificar el YAML a mano si quieres automatizar cambios controlados.
- Lo importante es que cada ajuste quede trazado, versionado y explicable.

**Qué decisión ayuda a tomar:** cómo pasar del análisis al contrato operativo sin perder reproducibilidad.

In [13]:
def update_settings_yaml(path, updates):
    with open(path, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    # Aplicar actualizaciones (diccionario anidado)
    def deep_update(d, u):
        for k, v in u.items():
            if isinstance(v, dict) and k in d:
                deep_update(d[k], v)
            else:
                d[k] = v
    deep_update(cfg, updates)
    with open(path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
# Ejemplo: update_settings_yaml('../conf/settings.yaml', {'case': {'test_size': 0.2}})

### Nota final — Versionado y trazabilidad de `settings.yaml`

> El valor de este notebook no termina en la recomendación. Termina cuando cada cambio queda documentado y puede volver a explicarse semanas después.

- Guarda cada versión relevante del YAML con nombre claro de experimento, fecha o propósito.
- Evita cambiar muchos parámetros a la vez si luego quieres entender qué produjo la mejora o el deterioro.
- Registra siempre el motivo del cambio: qué evidencia lo justificó y qué impacto esperabas observar.

**Idea central:** una buena parametrización no solo mejora métricas; también deja una historia auditable.

---

## 12) Resumen y siguientes pasos

> Si llegaste hasta aquí, ya no solo tienes resultados sueltos: tienes criterios para decidir qué escribir en `settings.yaml` y por qué.

### Qué deberías llevarte de este notebook

- Una recomendación razonada sobre cuántos datos necesitas.
- Una lista más defendible de variables para inferencia y segmentación.
- Una lectura más prudente sobre complejidad, outliers, validación temporal y valor esperado.
- Un criterio claro para fijar `retention_success_rate` y `clv_horizons`.

### Qué hacer después

1. Traslada al YAML solo las decisiones que puedas explicar con lógica de negocio y evidencia.
2. Guarda la versión del YAML usada en cada experimento.
3. Compara métricas antes y después de cada cambio para no mezclar varios ajustes a la vez.
4. Si una recomendación estadística contradice la operativa real, documenta el motivo y no automatices la decisión a ciegas.